In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
cmap = 'coolwarm'

In [ ]:
baseline = pd.read_csv('FeaturesBaseline.csv')
ride = pd.read_csv('FeaturesRide.csv')
fog=pd.read_csv('FeaturesFog.csv')


In [ ]:
# Convert 'Gender' to numeric if it exists
for df in [baseline, ride, fog]:
    if 'Gender' in df.columns:
        df['Gender'] = df['Gender'].map({'M': 0, 'F': 1})

# Then select only numeric columns
baseline_num = baseline.select_dtypes(include=[np.number])
ride_num     = ride.select_dtypes(include=[np.number])
fog_num = fog.select_dtypes(include=[np.number])
baseline_num.head()

In [ ]:
def basic_stats(df):
    return pd.DataFrame({
        'Mean': df.mean(),
        'Median': df.median(),
        'Min': df.min(),
        'Max': df.max(),
        'Std': df.std()
    })

baseline_stats = basic_stats(baseline_num)
ride_stats     = basic_stats(ride_num)
fog_stats      = basic_stats(fog_num)

# Display the result
print("=== Baseline statistics ===")
print(baseline_stats)
print("\n=== Ride statistics ===")
print(ride_stats)
print("\n=== Fog statistics ===")
print(fog_stats)

combined_stats = pd.concat(
    {'Baseline': baseline_stats, 'Ride': ride_stats, 'Fog': fog_stats},
    axis=1
)
combined_stats


In [ ]:
baseline_num = baseline.select_dtypes(include=[np.number])
ride_num = ride.select_dtypes(include=[np.number])
fog_num = fog.select_dtypes(include=[np.number])


In [ ]:
combined = pd.concat([baseline_num, ride_num, fog_num], ignore_index=True).dropna()
corr_combined = combined.corr()
corr_baseline = baseline_num.corr()
corr_ride = ride_num.corr()
corr_fog=fog_num.corr()
mask = np.triu(np.ones_like(corr_combined, dtype=bool))


In [ ]:
labels = [col.replace('_', ' ') for col in corr_baseline.columns]
annot = corr_baseline.copy()
annot[mask] = np.nan

In [ ]:
import matplotlib.pyplot as plt

variables = ['MeanSaccAmp', 'StdSaccAmp', 'MedianSaccAmp',
             'MeanPeakVelAmp', 'StdPeakVelAmp', 'MedianPeakVelAmp',
             'MeanSaccDur', 'StdSaccDur', 'MedianSaccDur',
             'MeanFixDur', 'StdFixDur', 'MedianFixDur',
             'NumSacc']

custom_titles = [
    'Mean Saccade Amplitude [°]',
    'SD Saccade Amplitude [°]',
    'Median Saccade Amplitude [°]',
    'Mean Peak Velocity Amplitude [°/s]',
    'SD Peak Velocity Amplitude [°/s]',
    'Median Peak Velocity Amplitude [°/s]',
    'Mean Saccade Duration [ms]',
    'SD Saccade Duration [ms]',
    'Median Saccade Duration [ms]',
    'Mean Fixation Duration [ms]',
    'SD Fixation Duration [ms]',
    'Median Fixation Duration [ms]',
    'Number of Saccades [a. u.]'
]

n_vars = len(variables)
n_cols = 3
n_rows = 5

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows * 4))
axes_flat = axes.flatten()

# prvo uklonimo sve prazne axes
for i in range(n_rows * n_cols):
    axes_flat[i].set_visible(False)

# crtamo varijable
for i, var in enumerate(variables):
    # poslednja varijabla ide u sredinu poslednjeg reda
    if i == n_vars-1:
        ax = axes_flat[(n_rows-1)*n_cols + n_cols//2]
    else:
        ax = axes_flat[i]
    
    ax.set_visible(True)
    
    data = [baseline[var].dropna(), ride[var].dropna(), fog[var].dropna()]
    positions = [1, 2, 3]
    
    bplot = ax.boxplot(
        data,
        positions=positions,
        widths=0.6,
        patch_artist=True,
        showfliers=True,
        flierprops=dict(marker='x', markerfacecolor='red', markeredgecolor='red', markersize=6),
        medianprops=dict(color='black')
    )
    
    colors = ["#000000", '#1a80bb', '#800074']
    for patch, color in zip(bplot['boxes'], colors):
        patch.set_facecolor('none')
        patch.set_edgecolor(color)
        patch.set_linewidth(2)
    
    ax.set_xticks(positions)
    ax.set_xticklabels(['Baseline', 'Ride', 'Fog'], fontsize=20)
    ax.tick_params(axis='y', labelsize=20)
    ax.set_title(custom_titles[i], fontsize=20)
    ax.grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.savefig("my_figure.png", dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
custom_titles = [
    'Mean Saccade Amplitude [°]',
    'SD Saccade Amplitude [°]',
    'Median Saccade Amplitude [°]',
    'Mean Peak Velocity Amplitude [°/s]',
    'SD Peak Velocity Amplitude [°/s]',
    'Median Peak Velocity Amplitude [°/s]',
    'Mean Saccade Duration [ms]',
    'SD Saccade Duration [ms]',
    'Median Saccade Duration [ms]',
    'Mean Fixation Duration [ms]',
    'SD Fixation Duration [ms]',
    'Median Fixation Duration [ms]',
    'Number of Saccades [a. u.]'
]
def plot_lower_triangle(corr_matrix, title, filename):
    
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool),k=1)
    plt.figure(figsize=(14, 12))
    sns.heatmap(
        corr_matrix, 
        mask=mask, 
        cmap=cmap, 
        annot=True, 
        fmt=".2f", 
        linewidths=.5,
        square=True, 
        cbar_kws={"shrink": .8}, 
        xticklabels=[label.replace('_', ' ') for label in custom_titles],
        yticklabels=[label.replace('_', ' ') for label in custom_titles]
    )
    plt.title(title, fontsize=16)
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig(filename, dpi=300)
    plt.show()
    plt.close()

# Plot and save
plot_lower_triangle(corr_combined, 'Correlation Matrix: Baseline + Ride + Fog+ Combined', 'Sacc_combined.jpg')
plot_lower_triangle(corr_baseline, 'Correlation Matrix: Baseline', 'Sacc_baseline.jpg')
plot_lower_triangle(corr_ride, 'Correlation Matrix: Ride', 'Sacc_ride.jpg')
plot_lower_triangle(corr_fog, 'Correlation Matrix: Fog', 'Sacc_fog.jpg')